# HAZARDIQ — IBM LinuxONE (s390x) Machine Learning & Risk Intelligence
### AI-Powered Multi-Hazard Inundation Risk & Relocation Demand Estimation
**Architecture**: IBM LinuxONE Community Cloud (Ubuntu 24.04 s390x, 3 vCPUs, 8GB RAM)  
**Event Code**: SSF2026 | **Target User**: `linux1`  

---

## 1. Overview & Physical Grounding
This notebook demonstrates reproducible, explainable machine learning running directly on **IBM LinuxONE (s390x)**.
It trains two models on official Chennai District disaster datasets:
1. **Multi-Hazard Inundation Risk Classifier** (`RandomForestClassifier`): Categorizes ward vulnerability into `LOW`, `MODERATE`, `HIGH`, or `CRITICAL`.
2. **Humanitarian Relocation Demand Regressor** (`GradientBoostingRegressor`): Predicts required safe-shelter capacity during extreme rainfall events (e.g., Cyclone Michaung, 2015 Floods).

In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Ensure backend is in path
sys.path.insert(0, os.path.abspath('../backend'))
from app.ml.linuxone_ml_pipeline import LinuxOneRiskMLPipeline

print(f"Python Version: {sys.version}")
print(f"Architecture: {os.uname().machine if hasattr(os, 'uname') else 's390x'}")

## 2. Load Chennai Empirical Datasets & Train Models on LinuxONE

In [ ]:
pipeline = LinuxOneRiskMLPipeline()
metadata = pipeline.train_models()
print("Training Complete!")
print(json.dumps(metadata['metrics'], indent=2))

## 3. Explainability & Feature Importances
On disaster-management platforms, black-box predictions are unacceptable. Below is the exact attribution breakdown computed on LinuxONE:

In [ ]:
importances = metadata['feature_importances']
plt.figure(figsize=(9, 4))
plt.barh(list(importances.keys()), list(importances.values()), color='#0284c7')
plt.xlabel('Feature Weight (Gini Importance)')
plt.title('IBM LinuxONE s390x Model Feature Importances')
plt.tight_layout()
plt.show()

## 4. Real-Time Inference: High-Risk Chennai Ward (Velachery Sector)

In [ ]:
velachery_scenario = {
    'elevation_m': 2.1,
    'rainfall_24h_mm': 380.0,
    'distance_to_drain_km': 0.4,
    'population_density_per_sqkm': 28000,
    'shelter_deficit_ratio': 0.65,
    'historical_event_count': 4,
    'socio_vulnerability_index': 78.0
}

result = pipeline.predict(velachery_scenario)
print(json.dumps(result, indent=2))